In [1]:
import pandas as pd
import numpy as np
import json
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('../data/nassau_candy_enriched.csv')
df['Order Date'] = pd.to_datetime(df['Order Date'])

print(f"Enriched data loaded: {df.shape[0]} rows, {df.shape[1]} columns")

Enriched data loaded: 6013 rows, 27 columns


### 1. Build Product Summary Table

In [3]:
product_summary = df.groupby(['Product Name', 'Division', 'Factory']).agg(
    Total_Revenue       = ('Sales', 'sum'),
    Total_Profit        = ('Gross Profit', 'sum'),
    Total_Cost          = ('Cost', 'sum'),
    Total_Units         = ('Units', 'sum'),
    Total_Orders        = ('Order ID', 'nunique'),
    Avg_Margin          = ('Gross_Margin_%', 'mean'),
    Avg_Profit_per_Unit = ('Profit_per_Unit', 'mean'),
    Avg_Cost_per_Unit   = ('Cost_per_Unit', 'mean')
).round(2).reset_index()

total_revenue = df['Sales'].sum()
total_profit  = df['Gross Profit'].sum()

product_summary['Revenue_Share_%'] = (product_summary['Total_Revenue'] / total_revenue * 100).round(2)
product_summary['Profit_Share_%']  = (product_summary['Total_Profit']  / total_profit  * 100).round(2)

print(f"Products summarized: {len(product_summary)}")

Products summarized: 15


### 2. Rank by Total Gross Profit

In [4]:
rank_by_profit = product_summary.sort_values('Total_Profit', ascending=False).reset_index(drop=True)
rank_by_profit.index += 1  # start rank from 1

print("RANKED BY TOTAL GROSS PROFIT")
print(rank_by_profit[['Product Name', 'Division', 'Total_Revenue', 
                       'Total_Profit', 'Profit_Share_%']].to_string())

RANKED BY TOTAL GROSS PROFIT
                         Product Name   Division  Total_Revenue  Total_Profit  Profit_Share_%
1      Wonka Bar -Scrumdiddlyumptious  Chocolate       16344.00      11350.00           20.53
2   Wonka Bar - Triple Dazzle Caramel  Chocolate       16680.00      10897.60           19.71
3   Wonka Bar - Nutty Crunch Surprise  Chocolate       14452.09      10311.09           18.65
4          Wonka Bar - Milk Chocolate  Chocolate       15499.25      10062.59           18.20
5           Wonka Bar - Fudge Mallows  Chocolate       14598.00       9732.00           17.60
6                  Lickable Wallpaper      Other        4960.00       2480.00            4.49
7                           Wonka Gum      Other         328.75        170.95            0.31
8              Everlasting Gobstopper      Sugar         130.00        104.00            0.19
9                         Hair Toffee      Sugar          76.50         59.50            0.11
10                          Kaz

### 3. Rank by Gross Margin %

In [5]:
rank_by_margin = product_summary.sort_values('Avg_Margin', ascending=False).reset_index(drop=True)
rank_by_margin.index += 1

print("RANKED BY GROSS MARGIN %")
print(rank_by_margin[['Product Name', 'Division', 'Avg_Margin', 
                       'Total_Revenue', 'Total_Profit']].to_string())

RANKED BY GROSS MARGIN %
                         Product Name   Division  Avg_Margin  Total_Revenue  Total_Profit
1              Everlasting Gobstopper      Sugar       80.00         130.00        104.00
2                         Hair Toffee      Sugar       77.78          76.50         59.50
3   Wonka Bar - Nutty Crunch Surprise  Chocolate       71.35       14452.09      10311.09
4      Wonka Bar -Scrumdiddlyumptious  Chocolate       69.44       16344.00      11350.00
5           Wonka Bar - Fudge Mallows  Chocolate       66.67       14598.00       9732.00
6   Wonka Bar - Triple Dazzle Caramel  Chocolate       65.33       16680.00      10897.60
7          Wonka Bar - Milk Chocolate  Chocolate       64.92       15499.25      10062.59
8                         Laffy Taffy      Sugar       62.31          31.84         19.84
9                Fizzy Lifting Drinks      Sugar       60.00          45.00         27.00
10                          Wonka Gum      Other       52.00         328.75

### 4. Rank by Profit per Unit

In [6]:
rank_by_ppu = product_summary.sort_values('Avg_Profit_per_Unit', ascending=False).reset_index(drop=True)
rank_by_ppu.index += 1

print("RANKED BY PROFIT PER UNIT")
print(rank_by_ppu[['Product Name', 'Division', 'Factory',
                    'Avg_Profit_per_Unit', 'Avg_Cost_per_Unit', 
                    'Total_Units']].to_string())

RANKED BY PROFIT PER UNIT
                         Product Name   Division            Factory  Avg_Profit_per_Unit  Avg_Cost_per_Unit  Total_Units
1                  Lickable Wallpaper      Other     Secret Factory                10.00              10.00          248
2              Everlasting Gobstopper      Sugar     Secret Factory                 8.00               2.00           13
3                         Hair Toffee      Sugar  The Other Factory                 3.50               1.00           17
4      Wonka Bar -Scrumdiddlyumptious  Chocolate      Lot's O' Nuts                 2.50               1.10         4540
5   Wonka Bar - Nutty Crunch Surprise  Chocolate      Lot's O' Nuts                 2.49               1.00         4141
6   Wonka Bar - Triple Dazzle Caramel  Chocolate    Wicked Choccy's                 2.45               1.30         4448
7           Wonka Bar - Fudge Mallows  Chocolate      Lot's O' Nuts                 2.40               1.20         4055
8     

### 5. Quadrant Classification

In [7]:
# Thresholds: median of total revenue and avg margin across products
# Median is more robust than mean — not skewed by extreme outliers

sales_median  = product_summary['Total_Revenue'].median()
margin_median = product_summary['Avg_Margin'].median()

print(f"Classification Thresholds:")
print(f"  Sales Median  : ${sales_median:,.2f}")
print(f"  Margin Median : {margin_median:.2f}%")
print()

def classify_quadrant(row):
    high_sales  = row['Total_Revenue'] >= sales_median
    high_margin = row['Avg_Margin']    >= margin_median

    if high_sales and high_margin:
        return 'STAR'
    elif not high_sales and high_margin:
        return 'HIDDEN GEM'
    elif high_sales and not high_margin:
        return 'VOLUME TRAP'
    else:
        return 'DEAD WEIGHT'

product_summary['Quadrant'] = product_summary.apply(classify_quadrant, axis=1)

print("QUADRANT CLASSIFICATION")
print(product_summary[['Product Name', 'Division', 'Total_Revenue', 
                        'Avg_Margin', 'Quadrant']].sort_values(
                        'Total_Revenue', ascending=False).to_string(index=False))

Classification Thresholds:
  Sales Median  : $328.75
  Margin Median : 62.31%

QUADRANT CLASSIFICATION
                     Product Name  Division  Total_Revenue  Avg_Margin    Quadrant
Wonka Bar - Triple Dazzle Caramel Chocolate       16680.00       65.33        STAR
   Wonka Bar -Scrumdiddlyumptious Chocolate       16344.00       69.44        STAR
       Wonka Bar - Milk Chocolate Chocolate       15499.25       64.92        STAR
        Wonka Bar - Fudge Mallows Chocolate       14598.00       66.67        STAR
Wonka Bar - Nutty Crunch Surprise Chocolate       14452.09       71.35        STAR
               Lickable Wallpaper     Other        4960.00       50.00 VOLUME TRAP
                        Kazookles     Other         617.50        7.69 VOLUME TRAP
                        Wonka Gum     Other         328.75       52.00 VOLUME TRAP
           Everlasting Gobstopper     Sugar         130.00       80.00  HIDDEN GEM
                      Hair Toffee     Sugar          76.50       77

### 6. Quadrant Summary

In [8]:
quadrant_summary = product_summary.groupby('Quadrant').agg(
    Num_Products    = ('Product Name', 'count'),
    Total_Revenue   = ('Total_Revenue', 'sum'),
    Total_Profit    = ('Total_Profit', 'sum'),
    Avg_Margin      = ('Avg_Margin', 'mean')
).round(2).reset_index()

quadrant_summary['Revenue_Share_%'] = (quadrant_summary['Total_Revenue'] / total_revenue * 100).round(2)
quadrant_summary['Profit_Share_%']  = (quadrant_summary['Total_Profit']  / total_profit  * 100).round(2)

print("QUADRANT SUMMARY")
print(quadrant_summary.to_string(index=False))

QUADRANT SUMMARY
   Quadrant  Num_Products  Total_Revenue  Total_Profit  Avg_Margin  Revenue_Share_%  Profit_Share_%
DEAD WEIGHT             4         109.50         56.30       48.34             0.13            0.10
 HIDDEN GEM             3         238.34        183.34       73.36             0.28            0.33
       STAR             5       77573.34      52353.28       67.54            92.54           94.69
VOLUME TRAP             3        5906.25       2698.45       36.56             7.05            4.88


### 7. Key Observations

In [9]:
stars        = product_summary[product_summary['Quadrant'] == 'STAR']
hidden_gems  = product_summary[product_summary['Quadrant'] == 'HIDDEN GEM']
volume_traps = product_summary[product_summary['Quadrant'] == 'VOLUME TRAP']
dead_weight  = product_summary[product_summary['Quadrant'] == 'DEAD WEIGHT']

print("KEY OBSERVATIONS")

print(f"\n⭐ STARS ({len(stars)} products)")
print(f"   These are your core business drivers")
for _, r in stars.iterrows():
    print(f"   → {r['Product Name']} | Margin: {r['Avg_Margin']:.1f}% | Profit: ${r['Total_Profit']:,.2f}")

print(f"\n💎 HIDDEN GEMS ({len(hidden_gems)} products)")
print(f"   High margin but low volume — opportunity to scale")
for _, r in hidden_gems.iterrows():
    print(f"   → {r['Product Name']} | Margin: {r['Avg_Margin']:.1f}% | Revenue: ${r['Total_Revenue']:,.2f}")

print(f"\n⚠️  VOLUME TRAPS ({len(volume_traps)} products)")
print(f"   Generating sales but margins are weak — needs pricing review")
for _, r in volume_traps.iterrows():
    print(f"   → {r['Product Name']} | Margin: {r['Avg_Margin']:.1f}% | Revenue: ${r['Total_Revenue']:,.2f}")

print(f"\n❌ DEAD WEIGHT ({len(dead_weight)} products)")
print(f"   Low sales AND low margins — discontinuation candidates")
for _, r in dead_weight.iterrows():
    print(f"   → {r['Product Name']} | Margin: {r['Avg_Margin']:.1f}% | Profit: ${r['Total_Profit']:,.2f}")

KEY OBSERVATIONS

⭐ STARS (5 products)
   These are your core business drivers
   → Wonka Bar - Fudge Mallows | Margin: 66.7% | Profit: $9,732.00
   → Wonka Bar - Milk Chocolate | Margin: 64.9% | Profit: $10,062.59
   → Wonka Bar - Nutty Crunch Surprise | Margin: 71.3% | Profit: $10,311.09
   → Wonka Bar - Triple Dazzle Caramel | Margin: 65.3% | Profit: $10,897.60
   → Wonka Bar -Scrumdiddlyumptious | Margin: 69.4% | Profit: $11,350.00

💎 HIDDEN GEMS (3 products)
   High margin but low volume — opportunity to scale
   → Everlasting Gobstopper | Margin: 80.0% | Revenue: $130.00
   → Hair Toffee | Margin: 77.8% | Revenue: $76.50
   → Laffy Taffy | Margin: 62.3% | Revenue: $31.84

⚠️  VOLUME TRAPS (3 products)
   Generating sales but margins are weak — needs pricing review
   → Kazookles | Margin: 7.7% | Revenue: $617.50
   → Lickable Wallpaper | Margin: 50.0% | Revenue: $4,960.00
   → Wonka Gum | Margin: 52.0% | Revenue: $328.75

❌ DEAD WEIGHT (4 products)
   Low sales AND low margins — 

### Save Results

In [10]:
product_analysis_report = {
    "thresholds": {
        "sales_median"  : round(sales_median, 2),
        "margin_median" : round(margin_median, 2),
        "note"          : "Median used over mean — more robust against revenue outliers"
    },
    "rankings": {
        "by_total_profit"    : rank_by_profit[['Product Name', 'Division', 
                                'Total_Profit', 'Profit_Share_%']].to_dict(orient='records'),
        "by_gross_margin"    : rank_by_margin[['Product Name', 'Division', 
                                'Avg_Margin']].to_dict(orient='records'),
        "by_profit_per_unit" : rank_by_ppu[['Product Name', 'Division', 
                                'Avg_Profit_per_Unit']].to_dict(orient='records')
    },
    "quadrant_classification": product_summary[['Product Name', 'Division', 'Factory',
                                'Total_Revenue', 'Total_Profit', 'Avg_Margin',
                                'Profit_Share_%', 'Quadrant']].to_dict(orient='records'),
    "quadrant_summary": quadrant_summary.to_dict(orient='records'),
    "key_observations": {
        "stars"        : stars['Product Name'].tolist(),
        "hidden_gems"  : hidden_gems['Product Name'].tolist(),
        "volume_traps" : volume_traps['Product Name'].tolist(),
        "dead_weight"  : dead_weight['Product Name'].tolist()
    }
}

report_path = '../outputs/reports/product_analysis_report.json'
os.makedirs(os.path.dirname(report_path), exist_ok=True)

with open(report_path, 'w') as f:
    json.dump(product_analysis_report, f, indent=4)

print(f"Product analysis report saved to: {report_path}")

Product analysis report saved to: ../outputs/reports/product_analysis_report.json


### Save Enriched Dataset with Quadrant Column

In [11]:
# Merge quadrant back into main df for use in future notebooks

df = df.merge(
    product_summary[['Product Name', 'Quadrant']],
    on='Product Name',
    how='left'
)

enriched_path = '../data/nassau_candy_enriched.csv'
df.to_csv(enriched_path, index=False)

print(f"Quadrant column added and enriched dataset updated")
print(f"Saved to: {enriched_path}")
print(f"\nQuadrant distribution:")
print(df['Quadrant'].value_counts())

Quadrant column added and enriched dataset updated
Saved to: ../data/nassau_candy_enriched.csv

Quadrant distribution:
Quadrant
STAR           5805
VOLUME TRAP     180
DEAD WEIGHT      15
HIDDEN GEM       13
Name: count, dtype: int64
